# 03 - Gold Aggregation

**Purpose of the Gold Layer:**  Build two business-ready Gold tables from Silver. One captures the current state of the network i.e. the latest snapshot per station, and the other summarises each station's behaviour across the full history and ranks them by investment priority. Gold converts that detailed snapshot history into station level outputs that answer questions such as:
- What is happening across the network right now?
- Which stations are most at risk?
- Which stations should be prioritised for operational review or investment?
- Which stations are frequently empty, full, or affected by dock issues?

**Source:** `workspace.bikepoint.silver_bikepoint_station`  
**Targets:**
- `workspace.bikepoint.gold_bikepoint_station_current_status` - one row per station, latest data only
- `workspace.bikepoint.gold_bikepoint_station_aggregated_metrics` - one row per station, aggregated across all data

**Grain shift**

Unlike the Bronze and Silver share grain which had **One row per BikePoint station per snapshot.**
Gold changes the grain. In Gold, each table is designed for a specific business purpose and has one clear meaning per row:

- `gold_bikepoint_station_current_status` keeps only the latest snapshot per station.
- `gold_bikepoint_station_aggregated_metrics` aggregates across all snapshots per station.

**Build Mode: Full Overwrite**

Both Gold tables are rebuilt from scratch on every run using `CREATE OR REPLACE TABLE`. This is different from the Silver layer, which uses incremental MERGE.

Full overwrite is suitable for Gold because:
- Gold tables are small, usually around one row per station.
- Gold metrics depend on the full Silver history.
- Rebuilding Gold ensures the latest Silver corrections are reflected immediately.

This keeps the Gold layer simple, reliable, and easy to explain.

### Business definitions

The Gold layer introduces business interpretation. Unlike Silver, which only performs parsing and row-level arithmetic, Gold creates decision-support metrics.

**Gold Table 1: Current Station Status**

The current status table gives the latest known state of every BikePoint station. It answers questions such as:

- Which stations are currently empty?
- Which stations are currently full?
- Which stations have low bike availability?
- Which stations have low return capacity?
- Which stations are locked or unavailable?

**Gold Table 2: Aggregated Station Metrics**

The aggregated metrics table is the strategic decision table. It summarises each station across all historical snapshots in Silver. It answers questions such as:

- Which stations are empty most often?
- Which stations are full most often?
- Which stations have the highest availability risk?
- Which stations have high occupancy volatility?
- Which stations should be prioritised for investment?
- How do stations behave during weekday, weekend, morning peak, and evening peak periods?

In [0]:
CATALOG = "workspace"
SCHEMA  = "bikepoint"

SILVER_TABLE = "silver_bikepoint"
GOLD_STATUS_TABLE  = "gold_bikepoint_station_current_status"
GOLD_METRICS_TABLE = "gold_bikepoint_station_aggregated_metrics"

SILVER_TABLE_FQN       = f"{CATALOG}.{SCHEMA}.{SILVER_TABLE}"
GOLD_STATUS_FQN        = f"{CATALOG}.{SCHEMA}.{GOLD_STATUS_TABLE}"
GOLD_METRICS_FQN       = f"{CATALOG}.{SCHEMA}.{GOLD_METRICS_TABLE}"

# Stress score weights
WEIGHT_EMPTY  = 0.35
WEIGHT_FULL   = 0.20
WEIGHT_BROKEN = 0.45

# Thresholds for the "current state" station_status label on Gold table 1
LOW_BIKES_THRESHOLD = 2  
LOW_DOCKS_THRESHOLD = 2  

print(f"Source:           {SILVER_TABLE_FQN}")
print(f"Target (status):  {GOLD_STATUS_FQN}")
print(f"Target (metrics): {GOLD_METRICS_FQN}")

total_weight = WEIGHT_EMPTY + WEIGHT_FULL + WEIGHT_BROKEN
if total_weight != 1.0:
    raise ValueError("Stress score weights must add up to 1.0")
print("Stress score weights are valid.")

## Read Silver

This section reads the Silver table and checks that it is ready to be used for Gold-level aggregation.

Before building Gold tables, we verify that:

- The Silver table exists
- It contains records
- It has more than one snapshot if historical analysis is expected
- Key fields such as `bikepoint_id`, `pipeline_run_id`, and `ingestion_ts` are available

This step is important because Gold depends fully on Silver. If Silver has missing data, zero rows, or only one snapshot, the Gold metrics may still run but the results will be less meaningful.

In [0]:
from pyspark.sql.utils import AnalysisException

try:
    silver_df = spark.table(SILVER_TABLE_FQN)
except AnalysisException:
    raise Exception(
        f"Silver table does not exist: {SILVER_TABLE_FQN}. "
        "Run 02_silver_transformation first."
    )

## Build `gold_bikepoint_station_current_status`

This table gives the latest available status for each BikePoint station. It is the operational “right now” view of the network.

The Silver table contains multiple snapshots for each station because every API run captures the state of all BikePoints at that time. For the current status Gold table, we only want the latest row for each station.

### How We Select the Latest Row

We use a SQL window function to rank each station’s snapshots from newest to oldest.

```sql
SELECT *
FROM silver_bikepoint_station
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY bikepoint_id
    ORDER BY ingestion_ts DESC
) = 1
```

**How this works:**
- `PARTITION BY bikepoint_id`: group rows by station, like SQL's `GROUP BY`, but for window functions
- `ORDER BY ingestion_ts DESC`: within each station's group, order by timestamp, newest first
- `ROW_NUMBER()`: number them, newest = 1, second-newest = 2, etc.
- `QUALIFY ... = 1` — filter to only the newest row per station

In [0]:
# Build the current-status Gold table.
# This table keeps only the latest snapshot for each BikePoint station.
build_status_sql = f"""
CREATE OR REPLACE TABLE {GOLD_STATUS_FQN}
USING DELTA
AS

SELECT
    -- Station identifiers and location details
    bikepoint_id,
    station_name,
    place_type,
    latitude,
    longitude,
    terminal_name,

    -- Latest bike and dock availability values
    nb_bikes,
    nb_empty_docks,
    nb_docks,
    nb_standard_bikes,
    nb_ebikes,
    total_active_docks,
    broken_dock_count,
    occupancy_rate,
    empty_rate,
    ebike_share,

    -- Operational flags from the Silver table
    is_empty,
    is_full,
    is_installed,
    is_locked,
    is_temporary,
    is_consistent,

    -- Current station status label
    CASE
        WHEN is_locked THEN 'locked'
        WHEN is_empty THEN 'empty'
        WHEN is_full THEN 'full'
        WHEN nb_bikes <= {LOW_BIKES_THRESHOLD} THEN 'low_bikes'
        WHEN nb_empty_docks <= {LOW_DOCKS_THRESHOLD} THEN 'low_docks'
        ELSE 'normal'
    END AS station_status,

    -- NTILE(3) would split the data into 3 equal groups
    CASE NTILE(3) OVER (ORDER BY nb_docks)
        WHEN 1 THEN 'small'
        WHEN 2 THEN 'medium'
        WHEN 3 THEN 'large'
    END AS capacity_category,

    -- Provenance
    ingestion_ts,
    pipeline_run_id

FROM {SILVER_TABLE_FQN}

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY bikepoint_id
    ORDER BY ingestion_ts DESC
) = 1

ORDER BY bikepoint_id
"""

spark.sql(build_status_sql) # Run the SQL to create or replace the Gold status table.
status_count = spark.table(GOLD_STATUS_FQN).count() # Count the rows to verify the table was built.
print(f"Built {GOLD_STATUS_FQN} with {status_count:,} rows")

In [0]:


spark.sql(f"""
    SELECT *
    FROM {GOLD_STATUS_FQN} LIMIT 5
""").display()

## Sanity check `gold_bikepoint_station_current_status`

Look at the distribution of `station_status` and `capacity_category` to confirm the labels are sensible.

In [0]:
print("Distribution of station_status\n")
spark.sql(f"""
    SELECT station_status, COUNT(*) AS count
    FROM {GOLD_STATUS_FQN}
    GROUP BY station_status
    ORDER BY count DESC
""").show(truncate=False)

In [0]:
print("\nDistribution of capacity_category\n")
spark.sql(f"""
    SELECT
        capacity_category,
        COUNT(*) AS count,
        MIN(nb_docks) AS min_docks,
        MAX(nb_docks) AS max_docks,
        ROUND(AVG(nb_docks), 1) AS avg_docks
    FROM {GOLD_STATUS_FQN}
    GROUP BY capacity_category
    ORDER BY MIN(nb_docks)
""").show(truncate=False)

## Build `gold_bikepoint_station_aggregated_metrics`

This table is the main analytical Gold table. It summarises the full Silver history into **one row per BikePoint station**.

While `gold_bikepoint_station_current_status` shows the latest state of each station, this table looks across all collected snapshots and measures how each station behaves over time.

### Why This Table Is Useful

This table helps answer strategic questions such as:

- Which stations are most often empty?
- Which stations are most often full?
- Which stations have unstable availability?
- Which stations may need more docking capacity?
- Which stations may need better bike rebalancing?
- Which stations should be prioritised for investment?

#### 1. Load the entire Silver table

In [0]:
from pyspark.sql import functions as func

silver_df = spark.table(SILVER_TABLE_FQN)
print(f"Silver rows: {silver_df.count():,}")

#### 2. Create Latest Station Dimension

In [0]:
'''This step gives us one row per station, with identifying info from
the latest detilas: station name, location, current dock count.'''

# A window function is a special SQL/PySpark function that lets you perform calculations across a group of rows while still keeping every original row.
# Similar to the GROUPBY Function but lets us keep the original rows.
from pyspark.sql.window import Window

# Define the window:
#   * partitionBy("bikepoint_id") groups all rows for one station together
#   * orderBy(ingestion_ts DESC) sorts within each group, newest first
w = Window.partitionBy("bikepoint_id").orderBy(func.col("ingestion_ts").desc())

# Build the station dimension table
station_dim_df = (
    silver_df
    .orderBy(func.col("bikepoint_id"), func.col("ingestion_ts").desc())
    .dropDuplicates(["bikepoint_id"])
    .select(
        "bikepoint_id",
        "station_name",
        "place_type",
        "latitude",
        "longitude",
        func.col("nb_docks").alias("current_nb_docks"),
    )
)
display(station_dim_df.limit(5))

#### 3. Aggregate across all snapshots per station

In [0]:
# These are the behavioural over time metrics, storing data one row per station.
# For each station, we are summarising behaviour across all snapshots, which tells us
# how variable that state is, not just one moment in time.
base_aggs_df = (
    silver_df
    .groupBy("bikepoint_id")
    .agg( # For each station, compute the summary metrics:
        # How many snapshots do we have for this station? 
        # Usefull to add confidence. A metric computed from 1 snapshot is much less reliable than one computed from 50.
        func.count("*").alias("snapshot_count"),

        # Occupancy statistics across all snapshots:
        func.round(
            func.avg("occupancy_rate"),2)
        .alias("avg_occupancy_rate"),
        func.round(
            func.min("occupancy_rate"),2)
        .alias("min_occupancy_rate"),
        func.round(
            func.max("occupancy_rate"),2)
        .alias("max_occupancy_rate"),
        #   How volatile is the station?
            # A station with avg=0.5 stddev=0.05 is more steady.
            # A station with avg=0.5 stddev=0.4 swings wildly between empty and full.
        func.round(
            func.stddev("occupancy_rate"),2)
        .alias("occupancy_volatility"), 
        

        # Percentage of time empty/full
        # is_empty/is_full is a boolean column with values True or False.
        # To calculate an average, we first convert the boolean into a number using .cast():
            # True  becomes 1.0
            # False becomes 0.0
        func.round(
            func.avg(func.col("is_empty").cast("double")),2)
        .alias("pct_time_empty"),
        func.round(
            func.avg(func.col("is_full").cast("double")),2)
        .alias("pct_time_full"),

        # The percentage of snapshots where a station had very few bikes available.
        # A station does not need to be completely empty to be risky. If a station often has only 1 or 2 bikes, users may still struggle to find a bike, especially during busy periods.
        func.round(
            func.avg(
                func.when(func.col("nb_bikes") <= LOW_BIKES_THRESHOLD, 1.0)
                    .otherwise(0.0)
            ), 2
        ).alias("pct_time_low_bikes"),

        # The percentage of snapshots where a station had very few empty docks available.
        # A station does not need to be completely full to create a poor user experience. If a station often has only 1 or 2 empty docks, users may be at risk of not being able to return a bike.
        func.round(
            func.avg(
                func.when(func.col("nb_empty_docks") <= LOW_DOCKS_THRESHOLD, 1.0)
                    .otherwise(0.0)
            ), 2
        ).alias("pct_time_low_docks"),

        # The percentage of snapshots where a station was locked or out of service.
        # This is different from empty/full problems. An empty station may need bike rebalancing. A full station may need bikes moved away. However, a locked/damaged station usually indicates a maintenance or operational issue.
        func.round(
            func.avg(func.col("is_locked").cast("double")), 2
        ).alias("pct_time_locked"),

        # E-bike availability 
        func.round(
            func.avg("ebike_share"),2)
        .alias("avg_ebike_share"),

        # Broken-dock rate: average fraction of docks that were broken, across snapshots. 
        func.round(
            func.avg(
                func.when(func.col("nb_docks") > 0, func.col("broken_dock_count") / func.col("nb_docks"))
                .otherwise(0) # If the condition is false, return 0.
        )
            ,2)
        .alias("broken_dock_rate"),

        func.max("broken_dock_count").alias("max_broken_observed"),
    )
)

display(base_aggs_df.limit(5))

#### 4. Time-based aggregations

In [0]:
# This section creates time-based metrics for each BikePoint station.
''' The aim is to compare how station occupancy behaves during different time periods,
such as weekdays, weekends, morning peak, and evening peak.
'''
time_sliced_df = (
    silver_df.groupBy("bikepoint_id").agg(
        # Weekday vs weekend occupancy
        func.round(
            func.avg(
                func.when(func.col("is_weekend") == False, func.col("occupancy_rate"))) # Average occupancy during weekdays only.
        ,2
        ).alias("weekday_avg_occupancy"),

         func.round(
            func.avg(
                func.when(func.col("is_weekend") == True, func.col("occupancy_rate")))
            ,2 # Average occupancy during weekends only.
        ).alias("weekend_avg_occupancy"),

        # Morning peak including London local hours between 7 and 9.
        func.round(
            func.avg(
                func.when(func.col("service_hour").between(7, 9), func.col("occupancy_rate"))) # Average occupancy during morning peak period.
            ,2
        ).alias("morning_peak_avg_occupancy"),

        # Evening peak including London local hours between 17 and 19.
        func.round(
            func.avg(
                func.when(func.col("service_hour").between(17, 19), func.col("occupancy_rate"))) # Average occupancy during the evening peak period.
            ,2
        ).alias("evening_peak_avg_occupancy"),

        # Sample-size counts (helps interpret the above averages)
        func.sum(func.when(func.col("is_weekend") == False, 1).otherwise(0))
            .alias("weekday_snapshot_count"), # Count how many weekday snapshots exist for each station.
        func.sum(func.when(func.col("is_weekend") == True, 1).otherwise(0))
            .alias("weekend_snapshot_count"), # Count how many weekend snapshots exist for each station.
    )
)

display(time_sliced_df.limit(5))

#### 5. Join everything together

In [0]:
combined_df = (
    base_aggs_df
    .join(time_sliced_df, on="bikepoint_id", how="inner")
    .join(station_dim_df, on="bikepoint_id", how="inner")
)

display(combined_df.limit(5))

#### 6. Compute Availability risk score, Capacity category, Risk band

**1. Availability risk score combines three operational risk signals:**
- pct_time_empty:
    - How often the station had zero bikes available.
    - High value = users often cannot hire a bike.
- pct_time_full:
    - How often the station had zero empty docks.
    - High value = users often cannot return a bike.

- broken_dock_rate:
    - Average share of docks that were not accounted for.
    - High value = possible maintenance or unavailable dock issue.

**Higher score = higher availability risk.**

**2. Capacity category:**
- This groups stations into small, medium, and large categories, based on their current number of docks.
- func.ntile(3) splits the sorted stations into 3 roughly equal groups:
    - 1 = stations with the lowest dock counts  -> small
    - 2 = stations with middle dock counts      -> medium
    - 3 = stations with highest dock counts     -> large

**3. Availability risk decile:**

This divides all stations into 10 roughly equal groups based on `availability_risk_score`. The stations are sorted from highest risk to lowest risk using:

`Window.orderBy(func.col("availability_risk_score").desc())`

Then `func.ntile(10)` assigns each station a decile from `1` to `10`.
- `1` = top 10% highest-risk stations
- `2` = next 10% highest-risk stations
- ...
- `10` = lowest-risk stations

This helps us compare stations by relative risk rather than only looking at the raw score.

**4. Availability risk band:**

The risk band converts the numeric decile into a simple readable category.

- Decile `1` → `critical`
- Deciles `2–3` → `high`
- Deciles `4–6` → `medium`
- Deciles `7–10` → `low`

Instead of saying a station is in decile `2`, we can say it has a `high` availability risk.

**5. Investment priority rank:**

`investment_priority_rank` gives each station an absolute rank based on `availability_risk_score`.

- Rank `1` = highest availability risk
- Higher rank number = lower priority

We use `func.rank()` instead of `func.row_number()` because `rank()` handles ties fairly.

In [0]:
# This step adds business-level metrics to the Gold table.
# These columns help us identify which BikePoint stations may need attention, rebalancing, maintenance, or future investment.

# For the Availability Risk Score, we have added weights that decide how important each signal is in the final score.
# availability_risk_score =
    #   0.35 × pct_time_empty
    # + 0.20 × pct_time_full
    # + 0.45 × broken_dock_rate
combined_df = combined_df.withColumn(
    "availability_risk_score",
    WEIGHT_EMPTY  * func.coalesce(func.col("pct_time_empty"),   func.lit(0)) # If the column is null, use 0 instead.
  + WEIGHT_FULL   * func.coalesce(func.col("pct_time_full"),    func.lit(0)) 
  + WEIGHT_BROKEN * func.coalesce(func.col("broken_dock_rate"), func.lit(0)) 
)

# Inverse of availability_risk_score. This is the same data, but more positive framing for dashboard.
combined_df = combined_df.withColumn(
    "reliability_score",
    func.lit(1.0) - func.col("availability_risk_score")
)

# Capacity category
capacity_window = Window.orderBy("current_nb_docks") # Window.orderBy("current_nb_docks") sorts all stations by dock count.
combined_df = combined_df.withColumn(
    "capacity_category",
    func.when(func.ntile(3).over(capacity_window) == 1, "small")
     .when(func.ntile(3).over(capacity_window) == 2, "medium")
     .otherwise("large")
)

# Availability risk decile helps us divide all stations into 10 groups based on their availability_risk_score
risk_window = Window.orderBy(func.col("availability_risk_score").desc())
combined_df = combined_df.withColumn( # withColumn add a new column or replace an existing column in a PySpark DataFrame 
    "availability_risk_decile",
    func.ntile(10).over(risk_window)
)

# Availability risk band converts the numeric risk decile into a readable category.
combined_df = combined_df.withColumn(
    "availability_risk_band",
    func.when(func.col("availability_risk_decile") == 1, "critical")
     .when(func.col("availability_risk_decile").between(2, 3), "high")
     .when(func.col("availability_risk_decile").between(4, 6), "medium")
     .otherwise("low")
)

# Investment priority rank gives each station an absolute rank based on availability risk
# Rank 1 = highest availability risk
# We use func.rank() instead of row_number() because rank handles ties fairly
combined_df = combined_df.withColumn(
    "investment_priority_rank",
    func.rank().over(risk_window)
)

display(combined_df.limit(5))

#### 7. Select Final Gold Metrics Columns and Write Table
This step prepares the final version of the `gold_bikepoint_station_metrics` table.

The final table includes:
- Station identifiers, such as `bikepoint_id`, `station_name`, and location
- Capacity context, such as dock count and capacity category
- Confidence indicators, such as snapshot counts
- Historical availability metrics, such as average occupancy and percentage of time empty/full
- Time-based metrics, such as weekday, weekend, morning peak, and evening peak occupancy
- Business metrics, such as availability risk score, risk band, and investment priority rank

In [0]:
# Select and order the final columns for the Gold metrics table.
final_df = combined_df.select(
    # Identifiers
    "bikepoint_id",
    "station_name",
    "place_type",
    "latitude",
    "longitude",

    # Capacity details
    func.col("current_nb_docks").alias("nb_docks"),
    "capacity_category",

    # Snapshot counts used to judge metric reliability
    "snapshot_count",
    "weekday_snapshot_count",
    "weekend_snapshot_count",

    # Historical availability metrics
    func.round("avg_occupancy_rate",    2).alias("avg_occupancy_rate"),
    func.round("min_occupancy_rate",    2).alias("min_occupancy_rate"),
    func.round("max_occupancy_rate",    2).alias("max_occupancy_rate"),
    func.round("occupancy_volatility",  2).alias("occupancy_volatility"),
    func.round("pct_time_empty",        2).alias("pct_time_empty"),
    func.round("pct_time_full",         2).alias("pct_time_full"),
    func.round("broken_dock_rate",      2).alias("broken_dock_rate"),
    "max_broken_observed",
    func.round("avg_ebike_share",       2).alias("avg_ebike_share"),
    func.round("pct_time_low_bikes",    2).alias("pct_time_low_bikes"),
    func.round("pct_time_low_docks",    2).alias("pct_time_low_docks"),
    func.round("pct_time_locked",       2).alias("pct_time_locked"),


    # Time-based occupancy metrics
    func.round("weekday_avg_occupancy",      2).alias("weekday_avg_occupancy"),
    func.round("weekend_avg_occupancy",      2).alias("weekend_avg_occupancy"),
    func.round("morning_peak_avg_occupancy", 2).alias("morning_peak_avg_occupancy"),
    func.round("evening_peak_avg_occupancy", 2).alias("evening_peak_avg_occupancy"),

    # Business risk and priority metrics
    func.round("reliability_score",       2).alias("reliability_score"),
    func.round("availability_risk_score", 2).alias("availability_risk_score"),
    "availability_risk_band",
    "investment_priority_rank",
).orderBy(func.col("availability_risk_score").desc())

# Write the Gold table using overwrite because it is fully rebuilt from Silver.
(
    final_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(GOLD_METRICS_FQN)
)

metrics_count = spark.table(GOLD_METRICS_FQN).count()
print(f"Built {GOLD_METRICS_FQN} with {metrics_count:,} rows")

In [0]:
spark.sql(f"""
    SELECT
        *
    FROM {GOLD_METRICS_FQN}
""").display()

## Sanity Check the Metrics Table

This section checks whether the Gold metrics table looks reasonable after it has been built.

These checks are not the final analysis. They are basic validation checks to make sure the table is usable before downstream reporting or recommendations.

We'll check how many stations fall into each `stress_band`

This helps confirm that the Gold table has been created correctly and that the scoring logic is producing plausible results.

In [0]:
print("Distribution of availability_risk_band:")
spark.sql(f"""
    SELECT
        availability_risk_band,
        -- Number of stations in each risk band
        COUNT(*) AS station_count,
        -- Average risk score for stations in this band
        ROUND(AVG(availability_risk_score), 2) AS avg_availability_risk_score,
        -- Average percentage of time stations were empty
        ROUND(AVG(pct_time_empty), 2) AS avg_pct_time_empty,
        -- Average percentage of time stations were full
        ROUND(AVG(pct_time_full), 2) AS avg_pct_time_full,
        -- Average broken/unavailable dock rate
        ROUND(AVG(broken_dock_rate), 2) AS avg_broken_dock_rate

    FROM {GOLD_METRICS_FQN}
    GROUP BY availability_risk_band
    -- Custom order so the highest-risk bands appear first
    ORDER BY
        CASE availability_risk_band
            WHEN 'critical' THEN 1
            WHEN 'high'     THEN 2
            WHEN 'medium'   THEN 3
            WHEN 'low'      THEN 4
        END
""").show(truncate=False)

In [0]:
%sql
-- Top 5 most at-risk stations
SELECT
    investment_priority_rank AS rank,
    bikepoint_id,
    station_name,
    capacity_category,
    availability_risk_band,
    reliability_score,
    ROUND(availability_risk_score, 3) AS risk,
    ROUND(pct_time_empty, 2) AS pct_empty,
    ROUND(pct_time_full, 2) AS pct_full,
    ROUND(broken_dock_rate, 2) AS broken_rate,
    snapshot_count
FROM workspace.bikepoint.gold_bikepoint_station_aggregated_metrics
ORDER BY availability_risk_score DESC
LIMIT 5;

In [0]:
%sql
-- Band distribution
SELECT
    availability_risk_band,
    COUNT(*) AS n,
    ROUND(AVG(availability_risk_score), 4) AS avg_risk
FROM workspace.bikepoint.gold_bikepoint_station_aggregated_metrics
GROUP BY availability_risk_band
ORDER BY
    CASE availability_risk_band
        WHEN 'critical' THEN 1
        WHEN 'high'     THEN 2
        WHEN 'medium'   THEN 3
        WHEN 'low'      THEN 4
    END;